In [0]:
%sql
INSERT INTO `p-cdh`.workflow.externalfilesconfiguration VALUES
('Moveo_files','p-cdh','`p-cdh`.externalfiles.extopt_out','OPT_OUT','Incremental',null,null,null,true);

INSERT INTO `p-cdh`.workflow.externalfilescolumnconfiguration VALUES
('OPT_OUT','0','0','TelephoneNumber','|','false','Windows-1253','false','true'),
('OPT_OUT','1','1','OptOut','|','false','Windows-1253','false','true'),
('OPT_OUT','2','2','OptOutDate','|','false','Windows-1253','false','true'),
('OPT_OUT','3','3','CampaignId','|','false','Windows-1253','false','true');

In [0]:
# Βρίσκει όλα τα views στα derivative/primary που είναι SELECT * (A/B/C)
# και εφαρμόζει WITH SCHEMA EVOLUTION.
# Πρώτα DRY RUN: τυπώνει τα statements χωρίς να τα τρέχει.

DRY_RUN = False #True  # άλλαξέ το σε False για να εκτελεστούν

rows = spark.sql("""
    SELECT table_schema, table_name,
           lower(regexp_replace(view_definition, '\\\\s+', ' ')) AS def
    FROM `p-cdh`.information_schema.views
    WHERE table_schema IN ('derivative','primary')
      AND lower(regexp_replace(view_definition, '\\\\s+', ' ')) RLIKE 'select \\\\* from'
    ORDER BY table_schema, table_name
""").collect()

statements = [
    f"ALTER VIEW `p-cdh`.{r['table_schema']}.{r['table_name']} WITH SCHEMA EVOLUTION"
    for r in rows
]

print(f"-- {len(statements)} views to alter\n")
for stmt in statements:
    print(stmt + ";")

if not DRY_RUN:
    print("\n--- EXECUTING ---")
    for stmt in statements:
        try:
            spark.sql(stmt)
            print(f"OK: {stmt}")
        except Exception as e:
            print(f"FAILED: {stmt}\n   -> {e}")


DNA26-1410(4)


In [0]:
%sql
-- Step 1: Enable column mapping (required for DROP COLUMN)
ALTER TABLE `p-cdh`.primaryarchive.interaction SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
ALTER TABLE `p-cdh`.primaryactive.casecode SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
ALTER TABLE `p-cdh`.primaryarchive.casecode SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
%sql
-- Step 2: Drop the columns
-- interaction: primaryarchive
ALTER TABLE `p-cdh`.primaryarchive.interaction DROP COLUMN LegalProcessStatusCode;
ALTER TABLE `p-cdh`.primaryarchive.interaction DROP COLUMN LegalProcessStatus;
ALTER TABLE `p-cdh`.primaryarchive.interaction DROP COLUMN LegalProcessStatusPriority;

-- casecode: primaryactive
ALTER TABLE `p-cdh`.primaryactive.casecode DROP COLUMN LPS;
ALTER TABLE `p-cdh`.primaryactive.casecode DROP COLUMN LPSStatus;
ALTER TABLE `p-cdh`.primaryactive.casecode DROP COLUMN LPSDate;

-- casecode: primaryarchive
ALTER TABLE `p-cdh`.primaryarchive.casecode DROP COLUMN LPS;
ALTER TABLE `p-cdh`.primaryarchive.casecode DROP COLUMN LPSStatus;
ALTER TABLE `p-cdh`.primaryarchive.casecode DROP COLUMN LPSDate;

In [0]:
%sql
CREATE OR REPLACE VIEW `p-cdh`.primary.interaction as
(SELECT
  a.* EXCEPT (BusinessDate),
  (
    SELECT
      MAX(BusinessDate)
    FROM
      `p-cdh`.primaryactive.interaction
  ) AS BusinessDate
FROM
  `p-cdh`.primaryarchive.interaction a
WHERE
  a.BusinessDate = DATE '2023-12-27'
UNION ALL
SELECT
  a.* EXCEPT (BusinessDate),
  (
    SELECT
      MAX(BusinessDate)
    FROM
      `p-cdh`.primaryactive.interaction
  ) AS BusinessDate
FROM
  `p-cdh`.primaryarchive.interaction a
WHERE
  a.BusinessDate = DATE '2026-06-04'
UNION ALL
SELECT
  *
FROM
  `p-cdh`.primaryactive.interaction)

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.portfolio
SET
    profitCenter = '98',
    AssetOwnerTaxID = '4805285TH',
    CollectionBankFile = 'Alpha'
WHERE Portfolioname LIKE 'Athena%';

BAUDE-149


In [0]:
%sql
-- trigger_calendar_day — μετά το trigger_business_day 
ALTER TABLE `p-cdh-delivery`.report_catalog.report_registry ADD COLUMN trigger_calendar_day INT COMMENT 'For monthly: calendar day of month to trigger (1-31). Takes precedence over trigger_business_day. NULL = use trigger_business_day.' AFTER trigger_business_day;
-- run_after_area — μετά το run_days 
ALTER TABLE `p-cdh-delivery`.report_catalog.report_registry ADD COLUMN run_after_area STRING COMMENT 'For monthly reports with run_days = "Eom": pipeline_area in pipeline.area_runs whose max business_date must equal CurrentMonthLastWorkingDate for the report to become eligible.' AFTER run_days;

INSERT INTO `p-cdh-delivery`.report_catalog.report_registry (report_name, job_name, frequency, trigger_business_day, trigger_calendar_day, run_days, run_after_area, is_active, execution_order, depends_on_report, depends_on_workflow, skip_email) VALUES ('AlphaBankGeneralli', 'AlphaBankGeneralli', 'monthly', NULL, NULL, 'Eom', 'derivative', TRUE, 10, NULL, 'Derivative Job Run - 3rdparties - Ptools', FALSE);